# Diagnóstico rápido de Gemini
Ejecuta las celdas en orden, desde Jupyter/VS Code con el entorno del proyecto o desde Colab.
Carga el `.env` local sin sobrescribir variables del entorno; fuera del repositorio solicita la clave de forma oculta.
No guarda credenciales en el notebook. Las llamadas de generación consumen cuota de tu cuenta.

El proyecto usa `gemini-3.8-flash` por defecto. Primero comprobamos el catálogo accesible y luego probamos
el modelo configurado, sin sustituirlo automáticamente si falla.

In [1]:
#%pip install -q "google-genai>=1.0" "python-dotenv>=1.0"

### 1. Cargar la misma configuración del proyecto
La prioridad de claves es `GEMINI_API_KEY` → `GOOGLE_API_KEY`, igual que en `LLM`.
Si cambias el `.env` después de ejecutar esta celda, reinicia el kernel para evitar conservar valores anteriores.

In [2]:
import os
from pathlib import Path
from getpass import getpass
from importlib.metadata import version
from dotenv import load_dotenv
from google import genai
from google.genai import types

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file() and (p / "src/automl_agents").is_dir()), None)
if root is not None:
    load_dotenv(root / ".env", override=False)

key_source = next((name for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY") if os.getenv(name)), None)
api_key = os.environ[key_source] if key_source else getpass("API key de Gemini (oculta): ").strip()
if not api_key:
    raise ValueError("Falta una API key de Gemini.")
MODEL_CONFIGURADO = os.getenv("AUTOML_LLM_MODEL", "gemini-3.8-flash").removeprefix("models/")
client = genai.Client(
    api_key=api_key, vertexai=False,
    http_options=types.HttpOptions(timeout=600_000, retry_options=types.HttpRetryOptions(attempts=1)),
)
print("google-genai:", version("google-genai"))
print("Origen de clave:", key_source or "entrada oculta")
print("Modelo configurado:", MODEL_CONFIGURADO)
print(".env del repositorio:", root / ".env" if root else "no localizado; se usa entorno/entrada oculta")

def diagnosticar(exc):
    code = getattr(exc, "code", None)
    message = str(exc).replace(api_key, "[CLAVE OCULTA]")
    print(f"{type(exc).__name__} | código={code}: {message[:1500]}")
    hints = {
        400: "Revisa la clave, los parámetros y la compatibilidad del modelo.",
        401: "Revisa la autenticación y la API key.",
        403: "Revisa permisos, restricciones de la clave y acceso al servicio.",
        404: "Revisa el nombre del modelo y su disponibilidad para esta clave/API.",
        429: "Revisa cuota, facturación y límites de solicitudes.",
    }
    print(hints.get(code, "Revisa el detalle del error; si es de red, conexión, proxy y timeout."))

google-genai: 2.28.0
Origen de clave: GOOGLE_API_KEY
Modelo configurado: gemini-3.5-flash-lite
.env del repositorio: d:\Tesis\tesis-automl-agents\.env


### 2. Consultar los modelos disponibles
Usa el catálogo devuelto por la API, según la [documentación oficial](https://ai.google.dev/api/models?hl=es-419).
Que esta consulta funcione confirma acceso al catálogo; la generación se comprueba por separado.

In [3]:
modelos = []
try:
    modelos = sorted({m.name.removeprefix("models/") for m in client.models.list()
                      if m.name and "generateContent" in (m.supported_actions or [])})
    print("Modelos que admiten generateContent:")
    print("\n".join(modelos) or "No se encontraron modelos de generación.")
    print("\n¿El modelo configurado aparece en el catálogo?", MODEL_CONFIGURADO in modelos)
except Exception as exc:
    diagnosticar(exc)

Modelos que admiten generateContent:
antigravity-preview-05-2026
antigravity-preview-09-2026
antigravity-preview-latest
deep-research-max-preview-04-2026
deep-research-preview-04-2026
deep-research-pro-preview-12-2025
gemini-2.5-computer-use-preview-10-2025
gemini-2.5-flash
gemini-2.5-flash-image
gemini-2.5-flash-lite
gemini-2.5-flash-preview-tts
gemini-2.5-pro
gemini-2.5-pro-preview-tts
gemini-3-flash-preview
gemini-3-pro-image
gemini-3-pro-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-image-preview
gemini-3.1-flash-lite
gemini-3.1-flash-lite-image
gemini-3.1-flash-lite-preview
gemini-3.1-flash-tts-preview
gemini-3.1-pro-preview
gemini-3.1-pro-preview-customtools
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-3.5-transcribe
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-3.8-flash-lite-tts
gemini-3.8-flash-tts
gemini-4-argon
gemini-flash-latest
gemini-flash-lite-latest
gemini-nano-banana-2.1
gemini-omni-1.1-flash
gemini-omni-flash-preview
gemini-pro-latest
gemini-robo

### 3. Probar texto y JSON
Primero ejecuta con el modelo configurado para reproducir el fallo. Si obtienes `404 / NotFound`,
copia un nombre del catálogo en `MODELO_PRUEBA` y vuelve a ejecutar esta celda.
Se realizan dos llamadas cortas si la primera funciona; los errores se muestran sin recurrir al generador por reglas.

In [4]:
import json
import time

MODELO_PRUEBA = MODEL_CONFIGURADO  # Puedes reemplazarlo por un nombre del catálogo.
# MODELO_PRUEBA = 'gemini-3.1-flash-lite'  # Puedes reemplazarlo por un nombre del catálogo.
texto_ok = json_ok = False
try:
    start = time.perf_counter()
    response = client.models.generate_content(
        model=MODELO_PRUEBA,
        contents="Responde únicamente: conexión OK",
        config=types.GenerateContentConfig(max_output_tokens=1024),
    )
    if not response.text:
        raise RuntimeError(f"Respuesta sin texto; revisa candidates y prompt_feedback: {response.candidates}")
    texto_ok = True
    print(f"Texto OK ({time.perf_counter() - start:.1f} s):", response.text)
    response = client.models.generate_content(
        model=MODELO_PRUEBA,
        contents='Devuelve un objeto JSON con la clave conexion y el valor "ok".',
        config=types.GenerateContentConfig(response_mime_type="application/json", max_output_tokens=1024),
    )
    data = json.loads(response.text or "")
    if not isinstance(data, dict) or data.get("conexion") != "ok":
        raise ValueError(f"JSON inesperado: {data}")
    json_ok = True
    print("JSON OK:", data)
except Exception as exc:
    diagnosticar(exc)

print(f"\nResultado: texto={texto_ok}, JSON={json_ok}, modelo={MODELO_PRUEBA}")
if texto_ok and json_ok:
    print(f"Para usar este modelo en los runs, configura en .env: AUTOML_LLM_MODEL={MODELO_PRUEBA}")
    print("Reinicia el proceso de la aplicación y revisa si el run tiene un llm_model explícito que prevalezca.")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Texto OK (1.4 s): conexión OK
JSON OK: {'conexion': 'ok'}

Resultado: texto=True, JSON=True, modelo=gemini-3.5-flash-lite
Para usar este modelo en los runs, configura en .env: AUTOML_LLM_MODEL=gemini-3.5-flash-lite
Reinicia el proceso de la aplicación y revisa si el run tiene un llm_model explícito que prevalezca.


**Cómo interpretar el resultado:** si el modelo original falla con `NotFound` y uno del catálogo funciona,
el nombre o la disponibilidad del modelo original es la causa probable. Si falla también el catálogo,
revisa el error de autenticación, acceso o red antes de cambiar el modelo.

Si texto y JSON funcionan pero el run sigue usando reglas, revisa el `llm_model` efectivo del run y el detalle
del error en sus artefactos. Esta prueba verifica la API y JSON básico; no valida toda la propuesta de criterios.
El notebook no modifica el `.env` ni la configuración de la aplicación. Antes de compartirlo, borra sus salidas.